In [1]:
# PHASE 11 — SELLER ANALYSIS

import pandas as pd
import numpy as np

# Load datasets
orders = pd.read_csv("../data/olist_orders_dataset.csv")
order_items = pd.read_csv("../data/olist_order_items_dataset.csv")
reviews = pd.read_csv("../data/olist_order_reviews_dataset.csv")

# Convert dates
date_columns = [
    "order_delivered_customer_date",
    "order_estimated_delivery_date"
]

for col in date_columns:
    orders[col] = pd.to_datetime(
        orders[col],
        errors="coerce"
    )

# ---------------------------------------------------------
# 1. Calculate delivery delay
# ---------------------------------------------------------

orders["delivery_delay_days"] = (
    orders["order_delivered_customer_date"]
    - orders["order_estimated_delivery_date"]
).dt.total_seconds() / (60 * 60 * 24)

orders["late_delivery_flag"] = (
    orders["delivery_delay_days"] > 0
).astype(int)

# ---------------------------------------------------------
# 2. Prepare order-level financial information
# ---------------------------------------------------------

order_financials = (
    order_items
    .groupby("order_id")
    .agg(
        revenue=("price", "sum"),
        freight_value=("freight_value", "sum")
    )
    .reset_index()
)

order_financials["freight_ratio"] = np.where(
    order_financials["revenue"] > 0,
    order_financials["freight_value"]
    / order_financials["revenue"] * 100,
    np.nan
)

# ---------------------------------------------------------
# 3. Connect orders to sellers
# ---------------------------------------------------------

seller_orders = (
    order_items[
        ["order_id", "seller_id"]
    ]
    .drop_duplicates()
)

seller_data = (
    seller_orders
    .merge(orders, on="order_id", how="left")
    .merge(order_financials, on="order_id", how="left")
    .merge(
        reviews[
            ["order_id", "review_score"]
        ],
        on="order_id",
        how="left"
    )
)

# Negative review = rating 1, 2 or 3
seller_data["negative_review_flag"] = (
    seller_data["review_score"] <= 3
).astype(int)

# ---------------------------------------------------------
# 4. Calculate seller-level performance metrics
# ---------------------------------------------------------

seller_performance = (
    seller_data
    .groupby("seller_id")
    .agg(
        orders=("order_id", "nunique"),
        revenue=("revenue", "sum"),
        average_rating=("review_score", "mean"),
        late_delivery_rate=("late_delivery_flag", "mean"),
        average_delivery_delay=("delivery_delay_days", "mean"),
        freight_ratio=("freight_ratio", "mean"),
        negative_review_rate=("negative_review_flag", "mean")
    )
    .reset_index()
)

# Convert rates to percentages
seller_performance["late_delivery_rate"] *= 100
seller_performance["negative_review_rate"] *= 100

# Round for readability
seller_performance = seller_performance.round({
    "revenue": 2,
    "average_rating": 2,
    "late_delivery_rate": 2,
    "average_delivery_delay": 2,
    "freight_ratio": 2,
    "negative_review_rate": 2
})

print("Number of sellers analyzed:",
      len(seller_performance))

print("\nSeller performance preview:")
print(seller_performance.head(10))

Number of sellers analyzed: 3095

Seller performance preview:
                          seller_id  orders   revenue  average_rating  \
0  0015a82c2db000af6aaaf3ae2ecb0532       3   2685.00            3.67   
1  001cca7ae9ae17fb1caed9dfb1094831     200  25162.93            3.98   
2  001e6ad469a905060d959994f1b41e4f       1    250.00            1.00   
3  002100f778ceb8431b7a1020ff7ab48f      51   1382.30            3.90   
4  003554e2dce176b5555353e4f3555ac8       1    120.00            5.00   
5  004c9cd9d87a3c30c522c48c4fc07416     158  20567.08            4.14   
6  00720abe85ba0859807595bbf045a33b      13   1053.40            3.62   
7  00ab3eff1b5192e5f1a63bcecfee11c8       1     98.00            5.00   
8  00d8b143d12632bad99c0ad66ad52825       1     86.00            5.00   
9  00ee68308b45bc5e2660cd833c3f81cc     135  20855.89            4.30   

   late_delivery_rate  average_delivery_delay  freight_ratio  \
0                0.00                  -15.59           2.35   
1     

In [2]:
# ---------------------------------------------------------
# 5. Create Seller Experience Score
# ---------------------------------------------------------

# Normalize rating to 0-100
rating_score = (
    seller_performance["average_rating"] / 5
) * 100

# Convert late-delivery rate into a positive performance score
delivery_score = (
    100 - seller_performance["late_delivery_rate"]
)

# Convert negative-review rate into a positive performance score
review_score = (
    100 - seller_performance["negative_review_rate"]
)

# Seller Experience Score
seller_performance["experience_score"] = (
    0.5 * rating_score
    + 0.3 * delivery_score
    + 0.2 * review_score
)

seller_performance["experience_score"] = (
    seller_performance["experience_score"].round(2)
)

# ---------------------------------------------------------
# 6. Define High / Low using median thresholds
# ---------------------------------------------------------

revenue_threshold = seller_performance["revenue"].median()
experience_threshold = seller_performance["experience_score"].median()

print("Revenue threshold (median):",
      round(revenue_threshold, 2))

print("Experience threshold (median):",
      round(experience_threshold, 2))

# ---------------------------------------------------------
# 7. Create seller segments
# ---------------------------------------------------------

seller_performance["revenue_level"] = np.where(
    seller_performance["revenue"] >= revenue_threshold,
    "High Revenue",
    "Low Revenue"
)

seller_performance["experience_level"] = np.where(
    seller_performance["experience_score"] >= experience_threshold,
    "High Experience",
    "Low Experience"
)

# ---------------------------------------------------------
# 8. Assign business segment
# ---------------------------------------------------------

def assign_segment(row):

    if (
        row["revenue_level"] == "High Revenue"
        and row["experience_level"] == "High Experience"
    ):
        return "Best Performers"

    elif (
        row["revenue_level"] == "High Revenue"
        and row["experience_level"] == "Low Experience"
    ):
        return "Priority Problem"

    elif (
        row["revenue_level"] == "Low Revenue"
        and row["experience_level"] == "High Experience"
    ):
        return "Growth Opportunity"

    else:
        return "Low Priority"


seller_performance["seller_segment"] = (
    seller_performance.apply(
        assign_segment,
        axis=1
    )
)

# ---------------------------------------------------------
# 9. Segment summary
# ---------------------------------------------------------

segment_summary = (
    seller_performance
    .groupby("seller_segment")
    .agg(
        seller_count=("seller_id", "count"),
        total_revenue=("revenue", "sum"),
        avg_orders=("orders", "mean"),
        avg_rating=("average_rating", "mean"),
        avg_late_delivery_rate=("late_delivery_rate", "mean"),
        avg_experience_score=("experience_score", "mean")
    )
    .reset_index()
)

segment_summary = segment_summary.round(2)

print("\nSeller Segment Summary:")
print(segment_summary)

Revenue threshold (median): 847.35
Experience threshold (median): 86.26

Seller Segment Summary:
       seller_segment  seller_count  total_revenue  avg_orders  avg_rating  \
0     Best Performers           677     4813414.70       46.78        4.48   
1  Growth Opportunity           869      244229.58        3.44        4.73   
2        Low Priority           678      210983.76        3.79        2.91   
3    Priority Problem           871     8677876.26       72.08        3.75   

   avg_late_delivery_rate  avg_experience_score  
0                    3.67                 91.48  
1                    1.00                 96.38  
2                   14.91                 63.13  
3                   11.69                 77.53  


In [3]:
# ---------------------------------------------------------
# 10. Identify Priority Problem Sellers
# ---------------------------------------------------------

priority_problem = (
    seller_performance[
        seller_performance["seller_segment"] == "Priority Problem"
    ]
    .sort_values(
        by="revenue",
        ascending=False
    )
)

print("Number of Priority Problem sellers:",
      len(priority_problem))

print("\nTop 20 Priority Problem Sellers:")

print(
    priority_problem[
        [
            "seller_id",
            "orders",
            "revenue",
            "average_rating",
            "late_delivery_rate",
            "average_delivery_delay",
            "freight_ratio",
            "negative_review_rate",
            "experience_score"
        ]
    ]
    .head(20)
)

Number of Priority Problem sellers: 871

Top 20 Priority Problem Sellers:
                             seller_id  orders    revenue  average_rating  \
857   4869f7a5dfa277a7dca6462dcf3b52b2    1132  232951.51            4.13   
1013  53243585a1d6dc2643021fd1853d8905     358  222776.05            4.13   
881   4a3ca9315b744ce9f8e9374361493884    1806  213271.28            3.83   
1535  7c67e1448b00f6e969d365cea6b010ab     982  190886.23            3.49   
2643  da8622b14eb17ae2831f4ac5b9dab84a    1314  167209.22            4.18   
192   1025f0e2d44d7041d6cf58b6550e0bfa     915  144068.65            3.99   
1824  955fee9216a65b617aa5c0531780ce60    1287  136440.96            4.16   
843   46dc3b2cc0980fb8ec44634e21d2718e     521  128330.69            4.19   
1235  6560211a19b47992c3666cc44a7e94c0    1854  125709.54            3.94   
1540  7d13fca15225358621be4086e1eb0964     565  116037.68            4.02   
1198  620c87c171fb2a6dd6e8bb4dec959fc6     740  115875.89            4.25   
11

In [4]:
# ---------------------------------------------------------
# 11. Business recommendations by seller segment
# ---------------------------------------------------------

segment_recommendations = {
    "Best Performers":
        "Maintain performance and use as benchmarks for other sellers.",

    "Priority Problem":
        "Prioritize for operational review, delivery improvement, "
        "and seller-performance intervention.",

    "Growth Opportunity":
        "Support these sellers with increased visibility, "
        "promotion, and order-growth opportunities.",

    "Low Priority":
        "Monitor performance and address major experience issues "
        "before investing heavily in growth."
}

for segment, recommendation in segment_recommendations.items():

    print("\n" + segment)
    print("-" * len(segment))
    print(recommendation)


Best Performers
---------------
Maintain performance and use as benchmarks for other sellers.

Priority Problem
----------------
Prioritize for operational review, delivery improvement, and seller-performance intervention.

Growth Opportunity
------------------
Support these sellers with increased visibility, promotion, and order-growth opportunities.

Low Priority
------------
Monitor performance and address major experience issues before investing heavily in growth.
